# 01 — Data quality check (BTS On-Time, October 2025)

**Question:** is the dataset clean enough to chain each aircraft's flights into daily rotations?

In [2]:
from pathlib import Path

import duckdb
import pandas as pd

# The notebook runs from notebooks/, so the project root is one level up
ROOT = Path.cwd().parent
PARQUET = ROOT / "data" / "parquet" / "*.parquet"

con = duckdb.connect()
con.execute(f"CREATE VIEW flights AS SELECT * FROM read_parquet('{PARQUET.as_posix()}')")

In [3]:
con.sql("""
    SELECT
        count(*)                    AS n_flights,
        min(flight_date)            AS first_day,
        max(flight_date)            AS last_day,
        count(DISTINCT carrier)     AS n_carriers,
        count(DISTINCT tail_number) AS n_aircraft,
        count(DISTINCT origin)      AS n_origin_airports
FROM flights
""").df()

,n_flights,first_day,last_day,n_carriers,n_aircraft,n_origin_airports
0,605844,2025-10-01,2025-10-31,14,5681,346


In [4]:
cols = con.sql("DESCRIBE flights").df()["column_name"]

query = "SELECT " + ", ".join(
    f"round(100.0 * avg(({c} IS NULL)::INT), 2) AS {c}" for c in cols
) + " FROM flights"

nulls = (
    con.sql(query).df()
    .T.rename(columns={0: "pct_null"})
    .sort_values("pct_null", ascending=False)
)
nulls

,pct_null
cancellation_code,99.47
late_aircraft_delay,79.82
nas_delay,79.82
security_delay,79.82
carrier_delay,79.82
weather_delay,79.82
arr_delay,0.71
actual_elapsed,0.71
air_time,0.71
wheels_on_hhmm,0.54


In [5]:
con.sql("""
    SELECT
        sum(cancelled::INT)                                              AS n_cancelled,
        sum(diverted::INT)                                               AS n_diverted,
        sum((arr_delay IS NULL)::INT)                                    AS n_arr_delay_null,
        sum((arr_delay IS NULL AND NOT cancelled AND NOT diverted)::INT) AS n_arr_null_unexplained,
        sum((arr_delay >= 15)::INT)                                      AS n_late_15,
        sum((late_aircraft_delay IS NOT NULL)::INT)                      AS n_with_cause,
        sum((tail_number IS NULL AND NOT cancelled)::INT)                AS n_no_tail_not_cancelled
    FROM flights
""").df().T

,0
n_cancelled,3204.0
n_diverted,1070.0
n_arr_delay_null,4274.0
n_arr_null_unexplained,0.0
n_late_15,122251.0
n_with_cause,122251.0
n_no_tail_not_cancelled,0.0


**Findings**
- Nulls are structural, not errors: arrival fields are empty only for cancelled/diverted flights (3,204 + 1,070 = 4,274, zero unexplained).
- Delay-cause columns are filled only when arrival delay ≥ 15 min (122,251 flights). A null means "under 15 min, not broken down", **not zero**.
- All 293 flights without a tail number are cancelled, so every operated flight can be assigned to an aircraft.
- Analysis scope: operated flights only (not cancelled, not diverted).